# Module 2 · Lecture 3 — Image Synthesis (VAE & GAN)

Hands-on companion to the *Lecture 3* slides. We build **two** generative models on
MedMNIST and compare them:

1. a **VAE** — stable, likelihood-based (MSE + KL), samples look *blurry*;
2. a **DCGAN** — adversarial, samples look *sharp* but training is unstable.

> Set `Runtime → GPU`. The VAE runs on free Colab; the **GAN is GPU-heavy →
> Colab Pro** for a full run. `QUICK_RUN = True` trains 1 epoch as a smoke test.

## 0 · Setup — MedMNIST

In [ ]:
%pip install -q medmnist

In [ ]:
import os, numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
import torchvision.transforms as T
import matplotlib.pyplot as plt
import medmnist; from medmnist import INFO

torch.manual_seed(0); np.random.seed(0)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)
QUICK_RUN = True

fig_dir = os.path.join('..', 'Figures'); os.makedirs(fig_dir, exist_ok=True)

In [ ]:
data_flag = 'pathmnist'   # any MedMNIST set works; try 'bloodmnist' too
info = INFO[data_flag]; C = info['n_channels']
DataClass = getattr(medmnist, info['python_class'])
print(f'{data_flag}: {C} channel(s)')

## 1 · VAE

Encoder → latent `z ~ N(mu, sigma)` → decoder. Loss = **reconstruction (MSE)** +
**KL**. The **reparameterization trick** `z = mu + sigma * eps` keeps sampling
differentiable.

In [ ]:
tfm = T.Compose([T.ToTensor()])   # pixels in [0,1] for MSE reconstruction
vae_ds = DataClass(split='train', transform=tfm, download=True)
vae_src = Subset(vae_ds, list(range(2000))) if QUICK_RUN else vae_ds
vae_loader = DataLoader(vae_src, batch_size=128, shuffle=True,
                        collate_fn=lambda b: torch.stack([x for x, _ in b]))
print('images:', len(vae_src))

In [ ]:
class VAE(nn.Module):
    def __init__(self, C=3, zdim=32, size=28):
        super().__init__()
        self.C, self.size, self.flat = C, size, C*size*size
        self.enc = nn.Sequential(nn.Flatten(), nn.Linear(self.flat, 256), nn.ReLU())
        self.mu, self.logvar = nn.Linear(256, zdim), nn.Linear(256, zdim)
        self.dec = nn.Sequential(nn.Linear(zdim, 256), nn.ReLU(),
                                 nn.Linear(256, self.flat), nn.Sigmoid())
    def reparam(self, mu, logvar):
        std = torch.exp(0.5*logvar)
        return mu + std*torch.randn_like(std)       # <-- reparameterization
    def forward(self, x):
        h = self.enc(x); mu, logvar = self.mu(h), self.logvar(h)
        z = self.reparam(mu, logvar)
        xr = self.dec(z).view(-1, self.C, self.size, self.size)
        return xr, mu, logvar

def vae_loss(xr, x, mu, logvar):
    recon = F.mse_loss(xr, x, reduction='sum')
    kl = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    return (recon + kl) / x.size(0)

In [ ]:
vae = VAE(C=C, zdim=32, size=28).to(device)
opt = torch.optim.Adam(vae.parameters(), lr=1e-3)
EPOCHS = 1 if QUICK_RUN else 10
for epoch in range(EPOCHS):
    vae.train(); running = 0.0
    for x in vae_loader:
        x = x.to(device)
        xr, mu, logvar = vae(x)
        loss = vae_loss(xr, x, mu, logvar)
        opt.zero_grad(); loss.backward(); opt.step()
        running += loss.item()
    print(f'VAE epoch {epoch+1}/{EPOCHS}: loss={running/len(vae_loader):.1f}')

### Sample new images from the prior and save for the slides

In [ ]:
vae.eval()
with torch.no_grad():
    z = torch.randn(12, 32, device=device)
    samples = vae.dec(z).view(-1, C, 28, 28).cpu()
fig, axes = plt.subplots(2, 6, figsize=(7, 2.4))
for ax, img in zip(axes.ravel(), samples):
    ax.imshow(img.permute(1, 2, 0).squeeze(), cmap='gray'); ax.axis('off')
plt.suptitle('VAE samples (blurry, stable)')
plt.tight_layout()
plt.savefig(os.path.join(fig_dir, 'vae_samples.png'), dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

## 2 · DCGAN

**Generator** (noise → image) vs. **Discriminator** (real vs. fake), trained
adversarially. We work at 32×32 with pixels normalized to `[-1, 1]` (the generator
ends in `tanh`).

In [ ]:
gan_tfm = T.Compose([T.Resize(32), T.ToTensor(),
                     T.Normalize([0.5]*C, [0.5]*C)])
gan_ds = DataClass(split='train', transform=gan_tfm, download=True)
gan_src = Subset(gan_ds, list(range(2000))) if QUICK_RUN else gan_ds
gan_loader = DataLoader(gan_src, batch_size=128, shuffle=True, drop_last=True,
                        collate_fn=lambda b: torch.stack([x for x, _ in b]))
ZDIM = 100

In [ ]:
class Generator(nn.Module):
    def __init__(self, zdim=ZDIM, C=3, ngf=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.ConvTranspose2d(zdim, ngf*4, 4, 1, 0), nn.BatchNorm2d(ngf*4), nn.ReLU(True),   # 4x4
            nn.ConvTranspose2d(ngf*4, ngf*2, 4, 2, 1), nn.BatchNorm2d(ngf*2), nn.ReLU(True),  # 8x8
            nn.ConvTranspose2d(ngf*2, ngf, 4, 2, 1), nn.BatchNorm2d(ngf), nn.ReLU(True),      # 16x16
            nn.ConvTranspose2d(ngf, C, 4, 2, 1), nn.Tanh())                                    # 32x32
    def forward(self, z):
        return self.net(z.view(z.size(0), -1, 1, 1))

class Discriminator(nn.Module):
    def __init__(self, C=3, ndf=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(C, ndf, 4, 2, 1), nn.LeakyReLU(0.2, True),                               # 16
            nn.Conv2d(ndf, ndf*2, 4, 2, 1), nn.BatchNorm2d(ndf*2), nn.LeakyReLU(0.2, True),    # 8
            nn.Conv2d(ndf*2, ndf*4, 4, 2, 1), nn.BatchNorm2d(ndf*4), nn.LeakyReLU(0.2, True),  # 4
            nn.Conv2d(ndf*4, 1, 4, 1, 0), nn.Sigmoid())                                        # 1
    def forward(self, x):
        return self.net(x).view(-1)

G = Generator(C=C).to(device)
D = Discriminator(C=C).to(device)
print('G/D params (M):', round(sum(p.numel() for p in G.parameters())/1e6, 2),
      round(sum(p.numel() for p in D.parameters())/1e6, 2))

In [ ]:
bce = nn.BCELoss()
optG = torch.optim.Adam(G.parameters(), lr=2e-4, betas=(0.5, 0.999))
optD = torch.optim.Adam(D.parameters(), lr=2e-4, betas=(0.5, 0.999))

EPOCHS = 1 if QUICK_RUN else 30
for epoch in range(EPOCHS):
    for x in gan_loader:
        x = x.to(device); b = x.size(0)
        real = torch.ones(b, device=device); fake = torch.zeros(b, device=device)
        # --- train D: real -> 1, fake -> 0 ---
        z = torch.randn(b, ZDIM, device=device)
        x_fake = G(z)
        d_loss = bce(D(x), real) + bce(D(x_fake.detach()), fake)
        optD.zero_grad(); d_loss.backward(); optD.step()
        # --- train G: fool D ---
        g_loss = bce(D(x_fake), real)
        optG.zero_grad(); g_loss.backward(); optG.step()
    print(f'GAN epoch {epoch+1}/{EPOCHS}: d_loss={d_loss.item():.3f} g_loss={g_loss.item():.3f}')

### Sample from the generator and save for the slides

In [ ]:
G.eval()
with torch.no_grad():
    z = torch.randn(12, ZDIM, device=device)
    samples = ((G(z).cpu() * 0.5) + 0.5).clamp(0, 1)   # denormalize [-1,1] -> [0,1]
fig, axes = plt.subplots(2, 6, figsize=(7, 2.4))
for ax, img in zip(axes.ravel(), samples):
    ax.imshow(img.permute(1, 2, 0).squeeze(), cmap='gray'); ax.axis('off')
plt.suptitle('GAN samples (sharper, less stable)')
plt.tight_layout()
plt.savefig(os.path.join(fig_dir, 'gan_samples.png'), dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

## Recap
- **VAE:** stable, principled (ELBO = MSE + KL), but samples are blurry.
- **GAN:** sharper samples, but unstable (mode collapse, non-convergence) — needs
  more epochs and the tricks from the slides (balanced LR, label smoothing).
- Evaluating generative models is hard (FID, downstream utility) — revisited in
  **Module 4**.
- **Try:** `QUICK_RUN = False`; increase `zdim`/epochs; compare the two sample grids.